# Le due fasi di una risposta, che costano in modo opposto

*Anatomia di un LLM* · capitolo 7, «Generare» · nel libro: §7.4 «Le due fasi di una risposta» · script `06-04`, `scripts/06_04_prefill_decode.py`

**Dove siamo.** Ultimo notebook: le due fasi di una risposta, leggere la domanda e scrivere la risposta, e perché costano in modo così diverso. I tempi dipendono dalla macchina; i rapporti fra le due fasi no.

Leggere la domanda e scrivere la risposta sono due lavori diversi. Il primo
elabora tutto insieme e satura il processore; il secondo procede un carattere
alla volta e passa il tempo ad aspettare la memoria. Quasi tutto quello che si
legge sui costi dei modelli discende da qui.

Si esegue con `python3 scripts/06_04_prefill_decode.py`
e dura circa 10 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

> **Questo notebook usa il modello addestrato.** Se `checkpoints/agogpt.pt` non c'è, la seconda cella ne scarica i pesi; per addestrarlo da te c'è `05_02_pretrain_agogpt`.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/06_04_prefill_decode.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [ ]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
# Questo notebook usa il modello addestrato. Se non c'e', si scaricano i
# suoi pesi (7 MB) invece di addestrarlo: per addestrarlo davvero c'e' il
# notebook 05_02_pretrain_agogpt, che dura alcuni minuti.
import urllib.request

modello = Path("checkpoints") / "agogpt.pt"
if not modello.exists():
    modello.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://github.com/fcollova/llm-anatomy/releases/download/modello-v0.1/agogpt-pesi.pt", modello)

In [ ]:
import time

import torch

from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.data import tokenizer_corpus
from llm_anatomy.model import AgoGPT
from llm_anatomy.train import carica_checkpoint
from llm_anatomy.utils import dec, numero, percorso, prepara, tabella, titolo, verifica

prepara()
torch.set_num_threads(1)
print("[misure] torch.set_num_threads(1) per rendere i tempi confrontabili")

file_ckpt = percorso("checkpoints", "agogpt.pt")
if not file_ckpt.exists():
    raise SystemExit("checkpoint assente: eseguire prima 05-02")
stato = carica_checkpoint(file_ckpt)
cfg = AgoGPTConfig(**stato["cfg"])
modello = AgoGPT(cfg)
modello.load_state_dict(stato["modello"])
modello.eval()
tok = tokenizer_corpus()
parametri = modello.numero_parametri()


def mediana(fn, ripetizioni: int = 9) -> float:
    with torch.no_grad():
        for _ in range(2):
            fn()
        tempi = []
        for _ in range(ripetizioni):
            t0 = time.perf_counter()
            fn()
            tempi.append(time.perf_counter() - t0)
    tempi.sort()
    return tempi[len(tempi) // 2]

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337
[misure] torch.set_num_threads(1) per rendere i tempi confrontabili


### Leggere la domanda, scrivere la risposta

> **Nel libro, §7.4 «Le due fasi di una risposta».** Perché leggere il testo di partenza e scrivere la risposta sono due lavori diversi?

**Cosa guardare.** Le due fasi, descritte.

In [ ]:
titolo("Leggere la domanda, scrivere la risposta")
print("  prefill: il prompt entra tutto insieme, ogni posizione e'")
print("           indipendente dalle altre nel calcolo delle proiezioni, e la")
print("           macchina lavora su matrici grandi")
print("  decode:  un token alla volta, matrici a una riga, e gli stessi pesi")
print("           riletti dalla memoria a ogni passo")


Leggere la domanda, scrivere la risposta
----------------------------------------
  prefill: il prompt entra tutto insieme, ogni posizione e'
           indipendente dalle altre nel calcolo delle proiezioni, e la
           macchina lavora su matrici grandi
  decode:  un token alla volta, matrici a una riga, e gli stessi pesi
           riletti dalla memoria a ogni passo


### Quanto rendono, a confronto

*Questa parte non è stampata nel libro.*

**Cosa guardare.** Il prefill elabora migliaia di token al secondo, il decode molti meno.

In [ ]:
titolo("Quanto rendono, a confronto")
righe = []
for lunghezza in (16, 64, 128):
    x = torch.randint(0, cfg.vocab_size, (1, lunghezza))
    t_prefill = mediana(lambda: modello(x))
    caches = modello.nuove_cache()
    with torch.no_grad():
        modello(x, caches=caches)
    uno = torch.randint(0, cfg.vocab_size, (1, 1))
    t_decode = mediana(lambda: modello(uno, caches=modello.nuove_cache()))
    righe.append([
        lunghezza,
        dec(1000 * t_prefill, 2),
        f"{lunghezza / t_prefill:,.0f}".replace(",", "."),
        dec(1000 * t_decode, 2),
        f"{1 / t_decode:,.0f}".replace(",", "."),
    ])
tabella(["prompt", "prefill (ms)", "token/s prefill", "decode (ms)",
         "token/s decode"], righe)

x = torch.randint(0, cfg.vocab_size, (1, 128))
uno = torch.randint(0, cfg.vocab_size, (1, 1))
t_prefill = mediana(lambda: modello(x))
t_decode = mediana(lambda: modello(uno, caches=modello.nuove_cache()))
print(f"  un token in prefill costa {dec(1000 * t_prefill / 128, 3)} ms,")
print(f"  un token in decode {dec(1000 * t_decode, 3)} ms: "
      f"{dec(t_decode / (t_prefill / 128), 1)} volte tanto")
verifica(t_decode > 5 * (t_prefill / 128),
         "un token generato costa molte volte un token letto")


Quanto rendono, a confronto
---------------------------


prompt  prefill (ms)  token/s prefill  decode (ms)  token/s decode
------  ------------  ---------------  -----------  --------------
16              1,95            8.219         0,83           1.203
64              3,67           17.452         0,78           1.278
128             6,07           21.076         0,87           1.152


  un token in prefill costa 0,049 ms,
  un token in decode 1,080 ms: 22,2 volte tanto
  [verifica superata] un token generato costa molte volte un token letto


### Le due attese che si notano

**Cosa guardare.** Il tempo al primo token e quello fra un token e l'altro.

In [ ]:
titolo("Le due attese che si notano")
prompt = 128
for da_generare in (20, 100):
    ttft = t_prefill
    tpot = t_decode
    totale = ttft + da_generare * tpot
    print(f"  prompt {prompt} token, generazione {da_generare} token:")
    print(f"    tempo al primo token (TTFT): {dec(1000 * ttft, 1)} ms")
    print(f"    tempo fra token (TPOT):      {dec(1000 * tpot, 1)} ms")
    print(f"    totale:                      {dec(totale, 3)} s, di cui "
          f"{dec(100 * ttft / totale, 1)}% di prefill")
print("  con prompt lunghi e risposte brevi domina il prefill, con prompt")
print("  brevi e risposte lunghe domina il decode: sono due problemi diversi")
print("  e si ottimizzano con tecniche diverse")


Le due attese che si notano
---------------------------
  prompt 128 token, generazione 20 token:
    tempo al primo token (TTFT): 6,2 ms
    tempo fra token (TPOT):      1,1 ms
    totale:                      0,028 s, di cui 22,4% di prefill
  prompt 128 token, generazione 100 token:
    tempo al primo token (TTFT): 6,2 ms
    tempo fra token (TPOT):      1,1 ms
    totale:                      0,114 s, di cui 5,4% di prefill
  con prompt lunghi e risposte brevi domina il prefill, con prompt
  brevi e risposte lunghe domina il decode: sono due problemi diversi
  e si ottimizzano con tecniche diverse


### Servire più utenti aiuta una fase sola

**Cosa guardare.** Il decode rende sempre di più col batch, il prefill no.

**Prova a cambiare:** aggiungi `128` ai batch.

<details><summary>Che cosa succede</summary>

Nel decode i token al secondo continuano a salire, nel prefill continuano a scendere: la differenza fra le due fasi si allarga.

</details>

In [ ]:
titolo("Servire piu' utenti aiuta una fase sola")
righe = []
for B in (1, 8, 32):
    xb = torch.randint(0, cfg.vocab_size, (B, 128))
    t_p = mediana(lambda: modello(xb), ripetizioni=5)
    unob = torch.randint(0, cfg.vocab_size, (B, 1))
    t_d = mediana(lambda: modello(unob, caches=modello.nuove_cache()))
    righe.append([B,
                  dec(1000 * t_p, 1),
                  numero(B * 128 / t_p),
                  dec(1000 * t_d, 2),
                  numero(B / t_d)])
tabella(["batch", "prefill (ms)", "token/s prefill", "decode (ms)",
         "token/s decode"], righe)
print("  il prefill era gia' saturo con una sola sequenza: il batch lo")
print("  rallenta quasi proporzionalmente. Il decode invece era in attesa")
print("  della memoria, e servire piu' sequenze insieme costa quasi lo stesso")
print("  tempo a passo: e' su questo che si regge il batching continuo dei")
print("  motori di servizio")


Servire piu' utenti aiuta una fase sola
---------------------------------------


batch  prefill (ms)  token/s prefill  decode (ms)  token/s decode
-----  ------------  ---------------  -----------  --------------
1               6,9           18.426         1,20             832
8              67,5           15.159         1,24           6.463
32            279,9           14.636         2,56          12.480
  il prefill era gia' saturo con una sola sequenza: il batch lo
  rallenta quasi proporzionalmente. Il decode invece era in attesa
  della memoria, e servire piu' sequenze insieme costa quasi lo stesso
  tempo a passo: e' su questo che si regge il batching continuo dei
  motori di servizio


### Byte letti per ogni carattere scritto

*Questa parte non è stampata nel libro.*

**Cosa guardare.** Ogni carattere scritto rilegge tutti i pesi del modello.

In [ ]:
titolo("Byte letti per ogni carattere scritto")
byte = parametri * 4
print(f"  parametri: {parametri:,}".replace(",", "."))
print(f"  byte letti a ogni passo di decode: {byte / 1e6:.2f} MB"
      .replace(".", ","))
print(f"  banda effettiva in decode: {byte / t_decode / 1e9:.2f} GB/s"
      .replace(".", ","))
print("  e' il conto che spiega perche' il decode di un modello grande e'")
print("  lento anche su hardware potente: si legge tutto il modello per")
print("  produrre un solo token")


Byte letti per ogni carattere scritto
-------------------------------------
  parametri: 1.789.056
  byte letti a ogni passo di decode: 7,16 MB
  banda effettiva in decode: 6,63 GB/s
  e' il conto che spiega perche' il decode di un modello grande e'
  lento anche su hardware potente: si legge tutto il modello per
  produrre un solo token
